# DataTrust — Tratamiento y curación de datos

## Contexto

Los notebooks 01, 02, 03 y 04 auditaron las tres tablas (`listings`, `reviews`, `calendar`) y consolidaron 14 hallazgos en una única tabla de gobernanza, con riesgo, prioridad y decisión propuesta para cada uno (notebook 04). Este notebook **ejecuta** esas decisiones en código: no vuelve a discutir si un hallazgo es o no un problema, eso ya se decidió; aquí se aplica la decisión ya justificada.

## Objetivo de este notebook

A partir de los CSV originales (`raw`), producir tres datasets curados (`listings_curado`, `reviews_curado`, `calendar_curado`) aplicando las decisiones de la tabla de gobernanza, y verificar con las mismas consultas SQL de control (huérfanos) usadas en la auditoría que el tratamiento funcionó, comparando antes y después.

Las decisiones de la tabla de gobernanza que fueron "mantener sin tratamiento" (R1, R3, R4, L6, L7, L8) también se documentan aquí explícitamente, para que quede constancia de que fueron una decisión deliberada y no un olvido.

Los datasets curados resultantes alimentan el notebook 06 (modelo) y el dashboard de Power BI.

## Nota metodológica: organización de este notebook

Igual que en los notebooks 01-03, el tratamiento se organiza tabla por tabla (`listings` → `reviews` → `calendar`), citando el identificador del hallazgo (L1-L8, R1-R4, C1-C2) definido en el notebook 04. Al final se añade una sección de verificación SQL antes/después y el guardado de los datasets curados.

**Convención de columnas añadidas**: cada columna imputada o derivada tiene un nombre explícito (ej. `price_imputado`, `license_categoria`) en vez de sobrescribir el dato original sin dejar rastro. Esto permite, en el notebook 06, decidir si una variable imputada debe tratarse igual que una observada, en vez de ocultar la diferencia.

In [1]:
import re
import pandas as pd
import sqlite3

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

listings = pd.read_csv("../data/airbnb_bcn/listings.csv", low_memory=False)
reviews = pd.read_csv("../data/airbnb_bcn/reviews.csv", low_memory=False)
calendar = pd.read_csv("../data/airbnb_bcn/calendar.csv", low_memory=False)

print("Filas (raw) -> listings:", listings.shape[0], "| reviews:", reviews.shape[0], "| calendar:", calendar.shape[0])

Filas (raw) -> listings: 15293 | reviews: 1033523 | calendar: 5623190


## 1. Tratamiento de `listings`

### 1.1 L1 — Eliminar columnas 100% vacías

**Decisión de gobernanza (prioridad Alta)**: eliminar las 12 columnas con 100% de valores nulos. No existe estrategia de imputación razonable para una columna completamente vacía, y mantenerla no aporta ninguna información al modelo ni al dashboard.

In [3]:
nulos_pct = listings.isna().mean() * 100
cols_vacias = nulos_pct[nulos_pct == 100.0].index.tolist()

listings_curado = listings.drop(columns=cols_vacias).copy()
print("Columnas eliminadas (L1):", len(cols_vacias))
print(cols_vacias)
print()
print("Columnas en listings_curado:", listings_curado.shape[1], "(antes:", listings.shape[1], ")")

Columnas eliminadas (L1): 12
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'calendar_updated', 'instant_bookable']

Columnas en listings_curado: 78 (antes: 90 )


**Resultado**: se eliminaron las 12 columnas identificadas en el notebook 01 (`host_since`, `host_response_rate`, `host_verifications`, `instant_bookable`, `neighborhood_overview`, entre otras), de 90 a 78 columnas. Ninguna fila se pierde en este paso, solo columnas sin información.

### 1.2 Variables de gobernanza derivadas (reutilizadas del notebook 01, con una corrección)

`host_profesional`, `license_categoria` y `license_problema` se definieron y justificaron en el notebook 01 sobre el `DataFrame` de auditoría, que no se guardaba en disco. Se recalculan aquí, con el mismo umbral (>50 anuncios).

**Corrección metodológica sobre la columna usada**: el notebook 01 (y el script `check_impacto_negocio_profesional.py`) usaban `host_listings_count` para este umbral. El diccionario de datos oficial de Inside Airbnb aclara que esa columna es *"el número de anuncios que tiene el anfitrión, según los cálculos de Airbnb"* — un conteo **global de la plataforma**, no limitado a Barcelona. La columna correcta para un estudio específico de Barcelona es `calculated_host_listings_count`, descrita como *"el número de anuncios que tiene el anfitrión en el scrape actual, en la geografía de la ciudad/región"* — sin nulos (87 en la otra) y calculada directamente sobre este snapshot. Se verificó el impacto del cambio: 478 anuncios (3,1%) cambian de clasificación, todos en la misma dirección (host con muchos anuncios fuera de Barcelona que dejan de contar como "profesional" aquí). El hallazgo de negocio no se debilita con la corrección, se fortalece (ver notebook 06 y ficha maestra).

In [4]:
def clasifica_licencia(valor):
    """Misma funcion y mismo criterio que en el notebook 01 (Seccion 3, ya corregida)."""
    if pd.isna(valor):
        return "Sin dato"
    valor = str(valor)
    if "Exempt" in valor:
        return "Exento (declarado)"
    if re.search(r"HUTB[-]?\d{4,}", valor, flags=re.IGNORECASE):
        return "Con numero HUTB (registro catalan)"
    if re.search(r"National registration number", valor, flags=re.IGNORECASE):
        return "Con numero NRUA (registro estatal, desde jul. 2025)"
    return "Otro / no clasificable"

listings_curado["host_profesional"] = listings_curado["calculated_host_listings_count"] > 50
listings_curado["license_categoria"] = listings_curado["license"].apply(clasifica_licencia)
listings_curado["license_problema"] = listings_curado["license_categoria"].isin(["Sin dato", "Otro / no clasificable"])

print("host_profesional:")
print(listings_curado["host_profesional"].value_counts())
print()
print("license_categoria:")
print(listings_curado["license_categoria"].value_counts())

host_profesional:
host_profesional
False    11728
True      3565
Name: count, dtype: int64

license_categoria:
license_categoria
Con numero HUTB (registro catalan)                     7194
Exento (declarado)                                     3924
Sin dato                                               3043
Con numero NRUA (registro estatal, desde jul. 2025)     948
Otro / no clasificable                                  184
Name: count, dtype: int64


### 1.3 L4 — Reformatear el campo `license`

**Decisión de gobernanza (prioridad Media)**: quitar las etiquetas HTML (`<br />`) y separar los números HUTB múltiples (nacional + autonómico) en una columna adicional. Es un problema de formato solucionable con reglas de texto: la información ya está presente, solo mal estructurada, no requiere inventar nada.

Se añaden dos columnas: `license_limpio` (texto sin HTML, con `|` como separador legible entre los fragmentos que antes separaba `<br />`) y `license_hutb_codigos` (los códigos HUTB encontrados, separados por `;`). El campo original `license` no se modifica, por trazabilidad.

In [5]:
def limpia_html(valor):
    if pd.isna(valor):
        return valor
    texto = re.sub(r"<br\s*/?>", " | ", str(valor), flags=re.IGNORECASE)
    texto = re.sub(r"<[^>]+>", "", texto)
    texto = re.sub(r"\s*\|\s*(\|\s*)+", " | ", texto)
    return texto.strip(" |").strip()


def extrae_hutb(valor):
    if pd.isna(valor):
        return ""
    return ";".join(re.findall(r"HUTB[-]?\d{4,}", str(valor), flags=re.IGNORECASE))


listings_curado["license_limpio"] = listings_curado["license"].apply(limpia_html)
listings_curado["license_hutb_codigos"] = listings_curado["license"].apply(extrae_hutb)

n_multi = (listings_curado["license_hutb_codigos"].str.count(";") >= 1).sum()
print("Registros con 2+ codigos HUTB separados en license_hutb_codigos:", n_multi)
print()
print("Ejemplo (license original vs. license_limpio vs. license_hutb_codigos):")
ejemplo = listings_curado.loc[
    listings_curado["license_hutb_codigos"].str.count(";") >= 1,
    ["license", "license_limpio", "license_hutb_codigos"],
].head(2)
for _, fila in ejemplo.iterrows():
    print("- original:", fila["license"][:90], "...")
    print("  limpio:  ", fila["license_limpio"])
    print("  codigos: ", fila["license_hutb_codigos"])

Registros con 2+ codigos HUTB separados en license_hutb_codigos: 5470

Ejemplo (license original vs. license_limpio vs. license_hutb_codigos):
- original: Spain - National registration number<br />ESFCTU000008058000039706000000000000000HUTB-0020 ...
  limpio:   Spain - National registration number | ESFCTU000008058000039706000000000000000HUTB-002062349 | Barcelona - Regional registration number | HUTB-002062
  codigos:  HUTB-002062349;HUTB-002062
- original: Spain - National registration number<br />ESFCTU000008106000547360000000000000000000HUTB00 ...
  limpio:   Spain - National registration number | ESFCTU000008106000547360000000000000000000HUTB0050572 | Barcelona - Regional registration number | HUTB-005057
  codigos:  HUTB0050572;HUTB-005057


**Resultado**: 5.470 registros (coincide exactamente con la cifra verificada en el notebook 01, ya corregida para detectar HUTB sin importar mayúsculas/minúsculas) quedan con sus dos códigos HUTB separados en `license_hutb_codigos`, y el texto en `license_limpio` ya no contiene etiquetas HTML sin procesar. Esto no cambia la clasificación de validez (`license_categoria`): un registro con dos HUTB concatenados ya se clasificaba como "Con número HUTB (registro catalán)" porque el patrón se detectaba igual con o sin HTML. El tratamiento aquí es de **formato y usabilidad del texto**, no de validez.

### 1.4 L2 — Imputación de nulos parciales

**Decisión de gobernanza (prioridad Media)**: imputación específica por columna, no una única estrategia genérica.

- **`price`**: mediana del grupo (`neighbourhood_cleansed`, `room_type`); si el grupo no tiene suficiente dato, mediana global. Precio depende sobre todo de zona y tipo de alojamiento.
- **`bedrooms`, `bathrooms`, `beds`**: mediana por `accommodates` (capacidad del alojamiento), con mediana global como respaldo. Son variables estructurales directamente relacionadas con cuánta gente se aloja.
- **`host_about`, `host_location`**: texto libre de perfil, se imputa con la categoría explícita `"Desconocido"` en vez de un promedio, porque no son variables numéricas.

Cada imputación deja una columna booleana `*_imputado` para no perder el rastro de qué valores son observados y cuáles son estimados.

In [6]:
# price: parsear a numerico y luego imputar
price_num = (
    listings_curado["price"].astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
listings_curado["price"] = pd.to_numeric(price_num, errors="coerce")
listings_curado["price_imputado"] = listings_curado["price"].isna()

mediana_por_grupo = listings_curado.groupby(["neighbourhood_cleansed", "room_type"])["price"].median()
mediana_global_price = listings_curado["price"].median()


def imputa_price(row):
    if pd.notna(row["price"]):
        return row["price"]
    valor = mediana_por_grupo.get((row["neighbourhood_cleansed"], row["room_type"]), pd.NA)
    return mediana_global_price if pd.isna(valor) else valor


listings_curado["price"] = listings_curado.apply(imputa_price, axis=1)
print(f"price: imputados {listings_curado['price_imputado'].sum()}, nulos restantes {listings_curado['price'].isna().sum()}")

# bedrooms / bathrooms / beds: mediana por capacidad (accommodates)
for col in ["bedrooms", "bathrooms", "beds"]:
    listings_curado[f"{col}_imputado"] = listings_curado[col].isna()
    mediana_cap = listings_curado.groupby("accommodates")[col].median()
    mediana_global = listings_curado[col].median()

    def imputa(row, col=col, mediana_cap=mediana_cap, mediana_global=mediana_global):
        if pd.notna(row[col]):
            return row[col]
        valor = mediana_cap.get(row["accommodates"], pd.NA)
        return mediana_global if pd.isna(valor) else valor

    listings_curado[col] = listings_curado.apply(imputa, axis=1)
    print(f"{col}: imputados {listings_curado[f'{col}_imputado'].sum()}, nulos restantes {listings_curado[col].isna().sum()}")

# host_about / host_location: categoria explicita "Desconocido"
for col in ["host_about", "host_location"]:
    listings_curado[f"{col}_imputado"] = listings_curado[col].isna()
    listings_curado[col] = listings_curado[col].fillna("Desconocido")
    print(f"{col}: imputados {listings_curado[f'{col}_imputado'].sum()}")

price: imputados 1938, nulos restantes 0


bedrooms: imputados 2562, nulos restantes 0


bathrooms: imputados 3565, nulos restantes 0


beds: imputados 2447, nulos restantes 0
host_about: imputados 4859
host_location: imputados 3098


**Resultado**: las cinco columnas numéricas/estructurales (`price`, `bedrooms`, `bathrooms`, `beds`) quedan con 0 nulos tras la imputación, y `host_about`/`host_location` quedan completas con la categoría `"Desconocido"` donde faltaba el dato. Los recuentos de imputados coinciden con los nulos verificados en el notebook 01 (1.938 en `price`, 2.562 en `bedrooms`, 3.565 en `bathrooms`, 2.447 en `beds`). Estas columnas no se usan todas como predictoras del modelo (`price` se excluye por fuga de información), pero `bedrooms`/`bathrooms`/`beds` sí, por lo que su imputación es necesaria antes del notebook 06.

### 1.5 L5 — `minimum_nights` > 365 días

**Decisión de gobernanza (prioridad Baja)**: revisar manualmente estos 3 casos, **no eliminar automáticamente**. El volumen es mínimo (3 de 15.293) y no hay evidencia de error: podría ser un alquiler de larga duración legítimo. Se marca con una columna booleana para que quede documentada la revisión, pero el valor no se modifica.

In [7]:
listings_curado["minimum_nights_revision_manual"] = listings_curado["minimum_nights"] > 365

print("Casos marcados para revision manual:", listings_curado["minimum_nights_revision_manual"].sum())
print(listings_curado.loc[
    listings_curado["minimum_nights_revision_manual"],
    ["id", "minimum_nights", "room_type", "accommodates"],
].to_string(index=False))

Casos marcados para revision manual: 3
      id  minimum_nights       room_type  accommodates
 7105437           730.0    Private room             1
13972983           900.0    Private room             2
28903054           500.0 Entire home/apt             4


**Resultado**: los 3 alojamientos (`id` 7105437, 13972983, 28903054) quedan marcados en `minimum_nights_revision_manual`, con su valor original intacto. No se ejecuta ninguna acción automática sobre ellos, tal como establece la decisión de gobernanza: la columna documenta que la revisión fue considerada y pospuesta a criterio humano, no que fue pasada por alto.

### 1.6 Tratamiento de la variable objetivo (`estimated_revenue_l365d`)

Esta decisión no forma parte de la tabla de 14 hallazgos L/R/C del notebook 04 (no es un problema de calidad de formato o integridad), es el **caso estrella de impacto en negocio** documentado en la ficha maestra del proyecto, verificado en `check_impacto_negocio_profesional.py`. Se incluye aquí porque `estimated_revenue_l365d` es el target del modelo (notebook 06) y su tratamiento debe quedar en el dataset curado, igual que cualquier otra decisión de curación.

**Causa raíz**: `estimated_revenue_l365d = 0` coincide siempre con `number_of_reviews_ltm = 0` (reseñas de los **últimos 365 días**), porque Inside Airbnb estima los ingresos a partir de esas reseñas recientes. Un alojamiento con estancia mínima larga (≥28 noches) puede estar ocupado y aun así no generar reseñas suficientes para que la fórmula estime nada, por lo que ese cero no es un ingreso real de cero, es "no estimable" con el método usado por la fuente.

**Corrección metodológica sobre la columna usada**: la primera versión de este análisis (en `check_impacto_negocio_profesional.py`) usaba `number_of_reviews` (histórico de toda la vida del anuncio) en vez de `number_of_reviews_ltm` (últimos 365 días). Es un desajuste de periodos: el target (`estimated_revenue_l365d`) es una métrica de 1 año, así que la variable de control también debe serlo. Un anuncio puede tener reseñas de hace 2 años (por eso el histórico no es cero) y ninguna reciente, y sufre el mismo problema de estimación. Verificado: con la columna correcta se capturan 791 anuncios más como "no estimable" (2.686 en vez de 1.895, un 42% más), y el hallazgo de negocio se mantiene (de hecho se fortalece, ver ficha maestra).

**Decisión de gobernanza**: marcar como `revenue_no_estimable` (no borrar, no imputar) los registros con la doble evidencia (`estimated_revenue_l365d == 0` + `number_of_reviews_ltm == 0` + `minimum_nights >= 28`), para no sobrecorregir excluyendo anuncios que de verdad rinden mal. La decisión de excluirlos o no de un análisis concreto se deja al notebook que lo use (el de negocio o el de modelo), según el propósito.

In [8]:
no_estimable = (
    listings_curado["estimated_revenue_l365d"].notna()
    & (listings_curado["estimated_revenue_l365d"] == 0)
    & (listings_curado["number_of_reviews_ltm"] == 0)
    & (listings_curado["minimum_nights"] >= 28)
)
listings_curado["revenue_no_estimable"] = no_estimable

tiene_revenue = listings_curado["estimated_revenue_l365d"].notna()
print("Anuncios marcados como revenue_no_estimable:", no_estimable.sum(),
      f"({no_estimable.sum() / tiene_revenue.sum() * 100:.1f}% de los que tienen dato de revenue)")

Anuncios marcados como revenue_no_estimable: 2686 (20.1% de los que tienen dato de revenue)


**Resultado**: 2.686 anuncios (20,1% de los que tienen dato de `estimated_revenue_l365d`) quedan marcados como `revenue_no_estimable`, coincidiendo con la cifra recalculada con la columna correcta (`number_of_reviews_ltm`) y verificada en la ficha maestra del proyecto. El valor de `estimated_revenue_l365d` no se modifica, solo se añade la bandera: así el notebook 06 puede decidir si excluye estos registros del entrenamiento (porque su "cero" no es informativo) o los conserva, sin perder la razón de esa decisión.

## 2. Tratamiento de `reviews`

### 2.1 R2 — Excluir reseñas huérfanas

**Decisión de gobernanza (prioridad Alta)**: excluir del dataset curado las filas cuyo `listing_id` no existe en `listings`. No hay forma de recuperar el alojamiento correspondiente; excluir es la única decisión defendible sin inventar un `listing_id`.

In [9]:
ids_listings = set(listings_curado["id"])
huerfanas_reviews = ~reviews["listing_id"].isin(ids_listings)

reviews_curado = reviews.loc[~huerfanas_reviews].copy()
print("Reviews huerfanas excluidas:", huerfanas_reviews.sum())
print("Filas reviews_curado:", reviews_curado.shape[0], "(antes:", reviews.shape[0], ")")

Reviews huerfanas excluidas: 6592
Filas reviews_curado: 1026931 (antes: 1033523 )


**Resultado**: se excluyen 6.592 reseñas (0,64% del total), coincidiendo con la cifra del notebook 02. `reviews_curado` queda con 1.026.931 filas, todas con un `listing_id` que existe en `listings_curado`.

### 2.2 R1, R3, R4 — Decisiones de "mantener sin tratamiento"

Estos tres hallazgos tuvieron prioridad de tratamiento Baja en el notebook 04 y la decisión explícita fue **no modificar la tabla**, documentarlos para una fase posterior si corresponde:

- **R1** (48.302 comentarios duplicados exactos): se mantiene sin tratamiento en esta fase; solo se filtrarían si se hiciera un análisis de texto/NLP específico, que no es el alcance de este proyecto.
- **R3** (512 casos de mismo `reviewer_id` con texto idéntico en alojamientos distintos): se mantiene sin tratamiento automático; la decisión fue documentarlo como señal para una eventual revisión manual de un equipo de confianza y seguridad, fuera del alcance de este proyecto.
- **R4** (9.917 reseñas con menos de 5 caracteres): se mantienen en la tabla completa, no son datos erróneos, solo menos informativas; se excluirían únicamente de un análisis de texto específico, que no existe en este proyecto.

`reviews_curado` contiene por tanto estos tres patrones sin alterar, igual que `reviews` original, salvo por la exclusión de R2 ya aplicada arriba.

## 3. Tratamiento de `calendar`

### 3.1 C1 — Excluir filas huérfanas

**Decisión de gobernanza (prioridad Alta)**: excluir los 113 `listing_id` (y sus filas) que no existen en `listings`, por la misma razón que R2: no se pueden vincular con los atributos correspondientes.

In [10]:
huerfanas_calendar = ~calendar["listing_id"].isin(ids_listings)

calendar_curado = calendar.loc[~huerfanas_calendar].copy()
print("Filas de calendar huerfanas excluidas:", huerfanas_calendar.sum(),
      "| listings distintos:", calendar.loc[huerfanas_calendar, "listing_id"].nunique())
print("Filas calendar_curado:", calendar_curado.shape[0], "(antes:", calendar.shape[0], ")")

Filas de calendar huerfanas excluidas: 41245 | listings distintos: 113
Filas calendar_curado: 5581945 (antes: 5623190 )


**Resultado**: se excluyen 41.245 filas (113 listings), coincidiendo exactamente con la cifra del notebook 03. `calendar_curado` queda con 5.581.945 filas, todas referidas a alojamientos que existen en `listings_curado`.

### 3.2 C2 — `minimum_nights` > `maximum_nights`

**Decisión de gobernanza (prioridad Alta, pero acción manual, no masiva)**: revisar manualmente los 6 alojamientos afectados, no aplicar una regla automática a las 746 filas. Es una contradicción lógica real, pero al estar concentrada en solo 6 anuncios es manejable caso por caso. Se marca con una columna booleana, igual que en L5, sin modificar los valores.

In [11]:
calendar_curado["rango_noches_incoherente"] = (
    calendar_curado["minimum_nights"] > calendar_curado["maximum_nights"]
)

print("Filas marcadas para revision manual:", calendar_curado["rango_noches_incoherente"].sum(),
      "| listings distintos:", calendar_curado.loc[calendar_curado["rango_noches_incoherente"], "listing_id"].nunique())

Filas marcadas para revision manual: 746 | listings distintos: 6


**Resultado**: 746 filas, 6 listings, quedan marcadas en `rango_noches_incoherente`, con sus valores originales intactos. Igual que en L5, la bandera documenta que la revisión se consideró y se dejó pendiente de criterio humano, no que se ignoró.

## 4. Verificación con SQL: antes y después de curar

**Criterio**: las mismas consultas SQL de control usadas en los notebooks 02 y 03 (`LEFT JOIN` contra `listings` para detectar huérfanos) se ejecutan sobre las tablas **raw** (ya cargadas en `database/airbnb.db`) y luego sobre las tablas **curadas**, para dejar evidencia reproducible de que el tratamiento funcionó, no solo una afirmación de que debería funcionar.

In [12]:
con = sqlite3.connect("../database/airbnb.db")

antes_reviews = pd.read_sql("""
SELECT COUNT(*) AS huerfanas FROM reviews r
LEFT JOIN listings l ON r.listing_id = l.id WHERE l.id IS NULL
""", con)
antes_calendar = pd.read_sql("""
SELECT COUNT(*) AS huerfanas FROM calendar c
LEFT JOIN listings l ON c.listing_id = l.id WHERE l.id IS NULL
""", con)

print("ANTES (tablas raw)   -> reviews huerfanas:", antes_reviews.iloc[0, 0],
      "| calendar huerfanas:", antes_calendar.iloc[0, 0])

listings_curado.to_sql("listings_curado", con, index=False, if_exists="replace")
reviews_curado.to_sql("reviews_curado", con, index=False, if_exists="replace")
calendar_curado.to_sql("calendar_curado", con, index=False, if_exists="replace")

despues_reviews = pd.read_sql("""
SELECT COUNT(*) AS huerfanas FROM reviews_curado r
LEFT JOIN listings_curado l ON r.listing_id = l.id WHERE l.id IS NULL
""", con)
despues_calendar = pd.read_sql("""
SELECT COUNT(*) AS huerfanas FROM calendar_curado c
LEFT JOIN listings_curado l ON c.listing_id = l.id WHERE l.id IS NULL
""", con)

print("DESPUES (tablas _curado) -> reviews huerfanas:", despues_reviews.iloc[0, 0],
      "| calendar huerfanas:", despues_calendar.iloc[0, 0])

con.close()

ANTES (tablas raw)   -> reviews huerfanas: 6592 | calendar huerfanas: 41245


DESPUES (tablas _curado) -> reviews huerfanas: 0 | calendar huerfanas: 0


**Resultado**: antes de curar, 6.592 reviews y 41.245 filas de calendar son huérfanas contra `listings`. Después de curar (`reviews_curado`, `calendar_curado` contra `listings_curado`), ambas cifras son 0. Las tres tablas curadas quedan además escritas en `database/airbnb.db`, disponibles para controles futuros con el mismo patrón SQL.

## 5. Guardar los datasets curados

Se guardan los tres datasets curados como CSV en `data/airbnb_bcn_curado/`, para que el notebook 06 (modelo) y el dashboard de Power BI puedan consumirlos directamente, sin depender de SQLite.

In [13]:
listings_curado.to_csv("../data/airbnb_bcn_curado/listings.csv", index=False)
reviews_curado.to_csv("../data/airbnb_bcn_curado/reviews.csv", index=False)
calendar_curado.to_csv("../data/airbnb_bcn_curado/calendar.csv", index=False)

print("Guardado en data/airbnb_bcn_curado/:")
print(" listings.csv:", listings_curado.shape)
print(" reviews.csv: ", reviews_curado.shape)
print(" calendar.csv:", calendar_curado.shape)

Guardado en data/airbnb_bcn_curado/:
 listings.csv: (15293, 91)
 reviews.csv:  (1026931, 6)
 calendar.csv: (5581945, 6)


## Resumen del tratamiento

| Tabla | Filas antes | Filas después | Columnas antes | Columnas después |
|---|---|---|---|---|
| `listings` | 15.293 | 15.293 (ninguna fila eliminada) | 90 | 91 (−12 vacías, +13 derivadas/imputación) |
| `reviews` | 1.033.523 | 1.026.931 (−6.592 huérfanas, R2) | 6 | 6 |
| `calendar` | 5.623.190 | 5.581.945 (−41.245 huérfanas, C1) | 5 | 6 (+1 bandera C2) |

**Decisiones ejecutadas con cambio de datos**: L1 (eliminar columnas), L2 (imputar price/bedrooms/bathrooms/beds/host_about/host_location), L4 (reformatear license), R2 (excluir huérfanas), C1 (excluir huérfanas), y el tratamiento de la variable objetivo (`revenue_no_estimable`).

**Decisiones ejecutadas como bandera, sin cambiar el valor original**: L5 y C2 (revisión manual).

**Decisiones de "mantener sin tratamiento", documentadas explícitamente**: L3 (license se conserva como variable de gobernanza), L6, L7, L8 (patrones derivados, no requieren acción propia), R1, R3, R4 (reviews).

Este notebook responde al objetivo 3 del proyecto ("aplicar un proceso de curación documentado y justificado"). El notebook 06 usa `listings_curado`, `reviews_curado` y `calendar_curado` para comparar las conclusiones de negocio y el rendimiento del modelo antes y después del tratamiento (objetivos 4 y 5).